In [27]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [61]:
BASE = r"C:\Users\gundo\OneDrive\Belgeler\airplanefailureprediction\data"

# For FD003 dataset
TRAIN_PATH = f"{BASE}\\train_FD003.csv"
TEST_PATH  = f"{BASE}\\test_FD003.csv"
RUL_PATH   = f"{BASE}\\RUL_FD003.txt"

LATENT_DIM = 8
BATCH_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 0.001

In [55]:
torch.cuda.is_available(), torch.cuda.device_count(),torch.cuda.current_device(), torch.cuda.get_device_name(0)

(True, 1, 0, 'NVIDIA A100-SXM4-40GB')

In [30]:
np.random.seed(42)
torch.manual_seed(42);

In [31]:
def load_data(train_path, test_path, rul_path):
   
    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    rul_values = pd.read_csv(rul_path, header=None, names=['RUL'])
    
    return train_df, test_df, rul_values

In [32]:
def add_rul_to_train(df):
    # Get max cycle for each unit
    max_cycles = df.groupby('unit_number')['time_in_cycles'].max().reset_index()
    max_cycles.columns = ['unit_number', 'max_cycle']
    
    # Merge and calculate RUL
    df = df.merge(max_cycles, on='unit_number')
    df['RUL'] = df['max_cycle'] - df['time_in_cycles']
    df.drop('max_cycle', axis=1, inplace=True)
    
    return df

In [33]:
def preprocess_data(train_df, test_df, sensor_cols, op_setting_cols, 
                    scaler_type='minmax', clip_rul=None):
    feature_cols = op_setting_cols + sensor_cols
    
    # Remove columns with zero variance (constant values)
    variance = train_df[feature_cols].var()
    valid_cols = variance[variance > 0.0001].index.tolist()
    print(f"Removed {len(feature_cols) - len(valid_cols)} constant columns")
    print(f"Using {len(valid_cols)} features: {valid_cols}")
    
    # Select scaler
    if scaler_type == 'minmax':
        scaler = MinMaxScaler(feature_range=(0, 1))
    else:
        scaler = StandardScaler()
    
    # Fit scaler on training data
    train_scaled = scaler.fit_transform(train_df[valid_cols])
    test_scaled = scaler.transform(test_df[valid_cols])
    
    # Apply RUL clipping if specified (piece-wise linear degradation)
    if clip_rul is not None and 'RUL' in train_df.columns:
        train_df['RUL'] = train_df['RUL'].clip(upper=clip_rul)
    
    return train_scaled, test_scaled, scaler, valid_cols


In [34]:
def create_sequences(data, sequence_length, stride=1):
    """
    Create sequences for time-series autoencoder.
    
    Parameters:
    -----------
    data : ndarray - Scaled sensor data
    sequence_length : int - Length of each sequence
    stride : int - Step between sequences
    
    Returns:
    --------
    sequences : ndarray of shape (n_samples, sequence_length, n_features)
    """
    sequences = []
    for i in range(0, len(data) - sequence_length + 1, stride):
        sequences.append(data[i:i + sequence_length])
    return np.array(sequences)

In [35]:
class DeepAutoencoder(nn.Module):
    """
    Deep Autoencoder with fully connected layers.
    
    Architecture:
    - Encoder: Input -> 128 -> 64 -> 32 -> latent_dim
    - Decoder: latent_dim -> 32 -> 64 -> 128 -> Input
    """
    
    def __init__(self, input_dim, latent_dim=8, dropout_rate=0.2):
        super(DeepAutoencoder, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(32, latent_dim),
            nn.ReLU()
        )
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, input_dim),
            nn.Sigmoid()  # Output in [0, 1] for MinMax scaled data
        )
    
    def forward(self, x):
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return reconstructed
    
    def encode(self, x):
        """Get latent representation."""
        return self.encoder(x)
    
    def decode(self, z):
        """Reconstruct from latent space."""
        return self.decoder(z)

In [36]:
class VariationalAutoencoder(nn.Module):
    """
    Variational Autoencoder (VAE) for probabilistic latent representations.
    
    Useful for:
    - Uncertainty quantification in health monitoring
    - Generating synthetic degradation patterns
    """
    
    def __init__(self, input_dim, latent_dim=8, dropout_rate=0.2):
        super(VariationalAutoencoder, self).__init__()
        
        self.input_dim = input_dim
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder_base = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
        )
        
        # Latent space parameters
        self.fc_mu = nn.Linear(32, latent_dim)
        self.fc_logvar = nn.Linear(32, latent_dim)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            
            nn.Linear(128, input_dim),
            nn.Sigmoid()
        )
    
    def encode(self, x):
        h = self.encoder_base(x)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick for backpropagation."""
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        reconstructed = self.decode(z)
        return reconstructed, mu, logvar

In [37]:
def vae_loss(reconstructed, original, mu, logvar, beta=1.0):
    """
    VAE loss = Reconstruction loss + KL divergence.
    
    Parameters:
    -----------
    beta : float - Weight for KL divergence (beta-VAE)
    """
    # Reconstruction loss (MSE)
    recon_loss = nn.MSELoss(reduction='sum')(reconstructed, original)
    
    # KL divergence
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    
    return recon_loss + beta * kl_loss

In [38]:
def train_autoencoder(model, train_loader, val_loader, num_epochs=100, 
                      learning_rate=0.001, device='cpu', model_type='ae',
                      early_stopping_patience=10):
    """
    Train the autoencoder model.
    
    Parameters:
    -----------
    model : nn.Module - Autoencoder model
    train_loader : DataLoader - Training data
    val_loader : DataLoader - Validation data
    num_epochs : int - Number of training epochs
    learning_rate : float - Learning rate
    device : str - 'cpu' or 'cuda'
    model_type : str - 'ae' for standard, 'vae' for variational
    early_stopping_patience : int - Epochs to wait before early stopping
    
    Returns:
    --------
    model, train_losses, val_losses
    """
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', 
                                                       factor=0.5, patience=5)
    
    if model_type == 'ae':
        criterion = nn.MSELoss()
    
    train_losses = []
    val_losses = []
    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None
    
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        for batch in train_loader:
            if isinstance(batch, (list, tuple)):
                data = batch[0].to(device)
            else:
                data = batch.to(device)
            
            optimizer.zero_grad()
            
            if model_type == 'vae':
                reconstructed, mu, logvar = model(data)
                loss = vae_loss(reconstructed, data, mu, logvar)
            else:
                reconstructed = model(data)
                loss = criterion(reconstructed, data)
            
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
        
        train_loss /= len(train_loader)
        train_losses.append(train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for batch in val_loader:
                if isinstance(batch, (list, tuple)):
                    data = batch[0].to(device)
                else:
                    data = batch.to(device)
                
                if model_type == 'vae':
                    reconstructed, mu, logvar = model(data)
                    loss = vae_loss(reconstructed, data, mu, logvar)
                else:
                    reconstructed = model(data)
                    loss = criterion(reconstructed, data)
                
                val_loss += loss.item()
        
        val_loss /= len(val_loader)
        val_losses.append(val_loss)
        
        # Learning rate scheduling
        scheduler.step(val_loss)
        
        # Early stopping check
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_model_state = model.state_dict().copy()
        else:
            patience_counter += 1
        
        if (epoch + 1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}] - "
                  f"Train Loss: {train_loss:.6f} - Val Loss: {val_loss:.6f}")
        
        if patience_counter >= early_stopping_patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    
    return model, train_losses, val_losses


In [39]:
def compute_reconstruction_error(model, data_loader, device='cpu', model_type='ae'):
    """
    Compute reconstruction error for anomaly detection.
    
    Returns per-sample MSE which can be used as a health indicator.
    """
    model.eval()
    errors = []
    
    with torch.no_grad():
        for batch in data_loader:
            if isinstance(batch, (list, tuple)):
                data = batch[0].to(device)
            else:
                data = batch.to(device)
            
            if model_type == 'vae':
                reconstructed, _, _ = model(data)
            else:
                reconstructed = model(data)
            
            # Per-sample MSE
            mse = ((data - reconstructed) ** 2).mean(dim=1)
            errors.extend(mse.cpu().numpy())
    
    return np.array(errors)

In [40]:
def extract_latent_features(model, data_loader, device='cpu', model_type='ae'):
    """
    Extract latent features for downstream tasks (e.g., RUL prediction).
    """
    model.eval()
    latent_features = []
    
    with torch.no_grad():
        for batch in data_loader:
            if isinstance(batch, (list, tuple)):
                data = batch[0].to(device)
            else:
                data = batch.to(device)
            
            if model_type == 'vae':
                mu, _ = model.encode(data)
                latent = mu
            else:
                latent = model.encode(data)
            
            latent_features.append(latent.cpu().numpy())
    
    return np.vstack(latent_features)

In [41]:
def plot_training_history(train_losses, val_losses, save_path=None):
    """Plot training and validation loss curves."""
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss', alpha=0.8)
    plt.plot(val_losses, label='Validation Loss', alpha=0.8)
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Autoencoder Training History')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

In [42]:
def plot_reconstruction_error_distribution(errors, save_path=None):
    """Plot distribution of reconstruction errors."""
    plt.figure(figsize=(10, 5))
    plt.hist(errors, bins=50, alpha=0.7, edgecolor='black')
    plt.xlabel('Reconstruction Error (MSE)')
    plt.ylabel('Frequency')
    plt.title('Distribution of Reconstruction Errors')
    plt.axvline(np.mean(errors), color='r', linestyle='--', label=f'Mean: {np.mean(errors):.4f}')
    plt.axvline(np.percentile(errors, 95), color='orange', linestyle='--', 
                label=f'95th percentile: {np.percentile(errors, 95):.4f}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

In [43]:
def plot_latent_space(latent_features, labels=None, save_path=None):
    """
    Visualize latent space using first 2 dimensions.
    """
    plt.figure(figsize=(10, 8))
    
    if labels is not None:
        scatter = plt.scatter(latent_features[:, 0], latent_features[:, 1], 
                             c=labels, cmap='viridis', alpha=0.5, s=10)
        plt.colorbar(scatter, label='RUL')
    else:
        plt.scatter(latent_features[:, 0], latent_features[:, 1], alpha=0.5, s=10)
    
    plt.xlabel('Latent Dimension 1')
    plt.ylabel('Latent Dimension 2')
    plt.title('Latent Space Visualization')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()

In [44]:
def plot_engine_degradation(train_df, model, scaler, valid_cols, 
                            unit_id, device='cpu', model_type='ae', save_path=None):
    """
    Plot reconstruction error over time for a single engine unit.
    This shows the degradation pattern detected by the autoencoder.
    """
    # Get data for specific unit
    unit_data = train_df[train_df['unit_number'] == unit_id].copy()
    unit_data = unit_data.sort_values('time_in_cycles')
    
    # Scale features
    unit_scaled = scaler.transform(unit_data[valid_cols])
    
    # Create DataLoader
    tensor_data = torch.FloatTensor(unit_scaled)
    loader = DataLoader(TensorDataset(tensor_data), batch_size=32, shuffle=False)
    
    # Compute reconstruction errors
    errors = compute_reconstruction_error(model, loader, device, model_type)
    
    # Plot
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
    
    cycles = unit_data['time_in_cycles'].values
    rul = unit_data['RUL'].values
    
    axes[0].plot(cycles, errors, 'b-', alpha=0.7)
    axes[0].set_ylabel('Reconstruction Error')
    axes[0].set_title(f'Engine Unit {unit_id} - Degradation Pattern')
    axes[0].grid(True, alpha=0.3)
    
    axes[1].plot(cycles, rul, 'r-', alpha=0.7)
    axes[1].set_xlabel('Time (Cycles)')
    axes[1].set_ylabel('Remaining Useful Life')
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


In [45]:
sensor_cols = [f'sensor_measurement_{i}' for i in range(1, 22)]
op_setting_cols = ['operational_setting_1', 'operational_setting_2', 'operational_setting_3']

In [62]:
print("\n--- Loading Data ---")
train_df, test_df, rul_df = load_data(TRAIN_PATH, TEST_PATH, RUL_PATH)
print(f"Training samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")
print(f"Number of training engines: {train_df['unit_number'].nunique()}")


--- Loading Data ---


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\gundo\\OneDrive\\Belgeler\\airplanefailureprediction\\data\\train_FD003.csv'

In [63]:
import os

BASE = r"C:\Users\gundo\OneDrive\Belgeler\airplanefailureprediction\data"

# Check if the base directory exists
if os.path.exists(BASE):
    print(f"✓ Base directory exists: {BASE}\n")
    print("Files in directory:")
    print("-" * 50)
    for file in os.listdir(BASE):
        print(f"  {file}")
else:
    print(f"✗ Base directory NOT found: {BASE}")
    
    # Try to find the correct path
    possible_paths = [
        r"C:\Users\gundo\OneDrive\Belgeler\airplanefailureprediction",
        r"C:\Users\gundo\OneDrive\Belgeler",
        r"C:\Users\gundo\Documents\airplanefailureprediction\data",
    ]
    
    print("\nSearching for your files...")
    for path in possible_paths:
        if os.path.exists(path):
            print(f"\n✓ Found directory: {path}")
            print("Contents:")
            for item in os.listdir(path):
                print(f"  {item}")

✗ Base directory NOT found: C:\Users\gundo\OneDrive\Belgeler\airplanefailureprediction\data

Searching for your files...


In [1]:
import os

BASE = r"C:\Users\gundo\airplanefailureprediction\data"

# FD001 dataset
DATASET = 'FD001'

TRAIN_PATH = os.path.join(BASE, f"train_{DATASET}.csv")
TEST_PATH  = os.path.join(BASE, f"test_{DATASET}.csv")
RUL_PATH   = os.path.join(BASE, f"RUL_{DATASET}.txt")

# Verify files exist
print("Checking files...")
for path in [TRAIN_PATH, TEST_PATH, RUL_PATH]:
    if os.path.exists(path):
        print(f"✓ Found: {path}")
    else:
        print(f"✗ Missing: {path}")

Checking files...
✗ Missing: C:\Users\gundo\airplanefailureprediction\data/train_FD001.csv
✗ Missing: C:\Users\gundo\airplanefailureprediction\data/test_FD001.csv
✗ Missing: C:\Users\gundo\airplanefailureprediction\data/RUL_FD001.txt


In [2]:
import os

BASE = r"C:\Users\gundo\airplanefailureprediction\data"

# Check if directory exists and list contents
if os.path.exists(BASE):
    print(f"✓ Directory exists: {BASE}\n")
    print("Files in directory:")
    print("-" * 50)
    for file in os.listdir(BASE):
        print(f"  {file}")
else:
    print(f"✗ Directory not found: {BASE}")

✗ Directory not found: C:\Users\gundo\airplanefailureprediction\data


In [3]:
import os

# Check each level of the path
paths_to_check = [
    r"C:\Users\gundo",
    r"C:\Users\gundo\airplanefailureprediction",
    r"C:\Users\gundo\airplanefailureprediction\data",
]

print("Checking path levels...")
print("-" * 50)

for path in paths_to_check:
    if os.path.exists(path):
        print(f"✓ EXISTS: {path}")
    else:
        print(f"✗ NOT FOUND: {path}")

# Also try to find the file directly
test_file = r"C:\Users\gundo\airplanefailureprediction\data\test_FD003.csv"
print("-" * 50)
if os.path.exists(test_file):
    print(f"✓ File exists: {test_file}")
else:
    print(f"✗ File not found: {test_file}")

Checking path levels...
--------------------------------------------------
✗ NOT FOUND: C:\Users\gundo
✗ NOT FOUND: C:\Users\gundo\airplanefailureprediction
✗ NOT FOUND: C:\Users\gundo\airplanefailureprediction\data
--------------------------------------------------
✗ File not found: C:\Users\gundo\airplanefailureprediction\data\test_FD003.csv
